# Ridge models with Palmer Penguins


## 1. Setup
If needed, uncomment and run the installation line once. Restart the kernel after installation if requested. `palmerpenguins` includes the data, so loading it does not require a separate data download.

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from palmerpenguins import load_penguins
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, RidgeClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
)



### Load data


In [ ]:
penguins = load_penguins().dropna()
penguins.head()

## Predict body mass


In [ ]:
mass_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
X_mass = penguins[mass_features]
y_mass = penguins["body_mass_g"]

### Train-Test Split

In [ ]:
X_mass_train, X_mass_test, y_mass_train, y_mass_test = train_test_split(
    X_mass, y_mass, test_size=0.20, random_state=909
)


### Step 3 — Define the preprocessing and model pipeline
1. `SimpleImputer`: replace missing predictor values with training medians.
2. `StandardScaler`: center each predictor and scale to unit standard deviation.
3. `Ridge`: fit a model with a starting penalty of `alpha=1`.

Scaling matters because ridge penalizes coefficient magnitudes. Without scaling, measurement units affect the penalty. The pipeline applies the same learned transformations to future observations. Defining the pipeline does not fit it.

In [ ]:
mass_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0)),
])

### Step 4 — Fit the model
`.fit(X, y)` learns the medians, scaling parameters, and regression coefficients from the training set.

In [ ]:
mass_model.fit(X_mass_train, y_mass_train)

### Step 5 — Make predictions
`.predict()` accepts predictors without the target and returns predicted body mass in grams.

In [ ]:
mass_predictions = mass_model.predict(X_mass_test)
display(pd.DataFrame({
    "observed_g": y_mass_test,
    "predicted_g": mass_predictions,
    "residual_g": y_mass_test - mass_predictions,
}).head().round(1))

### Step 6 — Evaluate against a simple baseline
RMSE emphasizes larger errors; MAE is the mean absolute error. Both are in grams. $R^2=1$ indicates perfect predictions, and negative values are possible. The baseline always predicts the **training mean**. A useful model should improve on this baseline.

In [ ]:
mass_baseline = DummyRegressor(strategy="mean")
mass_baseline.fit(X_mass_train, y_mass_train)
baseline_mass_predictions = mass_baseline.predict(X_mass_test)

def regression_metrics(y_true, y_pred):
    return {
        "RMSE_g": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE_g": mean_absolute_error(y_true, y_pred),
        "R_squared": r2_score(y_true, y_pred),
    }

display(pd.DataFrame({
    "Training-mean baseline": regression_metrics(y_mass_test, baseline_mass_predictions),
    "Ridge (alpha=1)": regression_metrics(y_mass_test, mass_predictions),
}).T.round(3))

### Step 7 — Inspect predictions and coefficients
Points near the diagonal have similar observed and predicted mass. Each coefficient describes a one-training-standard-deviation increase in that predictor, holding the other predictors fixed. These are predictive associations, not causal effects.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(y_mass_test, mass_predictions, alpha=0.75)
limits = [min(y_mass_test.min(), mass_predictions.min()),
          max(y_mass_test.max(), mass_predictions.max())]
ax.plot(limits, limits, "k--", label="Perfect prediction")
ax.set(xlabel="Observed body mass (g)", ylabel="Predicted body mass (g)",
       title="Ridge regression: test predictions")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
mass_coefficients = pd.Series(
    mass_model.named_steps["ridge"].coef_,
    index=mass_features,
    name="grams_per_predictor_standard_deviation",
)
display(mass_coefficients.round(2))
print("Intercept (grams):", round(mass_model.named_steps["ridge"].intercept_, 2))

## 3. Example two: predict species
### Step 1 — Choose predictors and target
Species is categorical, so use `RidgeClassifier`. Keep the species labels as strings; do not assign arbitrary numeric labels and fit ordinary `Ridge`. Body mass is now a valid predictor because species is the target. This assumes body mass will be available when making a new species prediction.

In [ ]:
species_features = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"
]
species_data = penguins.dropna(subset=["species"]).copy()
X_species = species_data[species_features]
y_species = species_data["species"]
display(y_species.value_counts().rename("count"))

### Step 2 — Create a stratified train/test split
`stratify=y_species` approximately preserves the class proportions in both subsets. This is especially useful when class sizes differ.

In [ ]:
X_species_train, X_species_test, y_species_train, y_species_test = train_test_split(
    X_species, y_species, test_size=0.20,
    random_state=RANDOM_STATE, stratify=y_species,
)
display(pd.DataFrame({
    "train_proportion": y_species_train.value_counts(normalize=True),
    "test_proportion": y_species_test.value_counts(normalize=True),
}).round(3))

### Step 3 — Define a classification pipeline
The preprocessing steps are the same as before. Only the final estimator changes.

In [ ]:
species_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", RidgeClassifier(alpha=1.0)),
])

### Step 4 — Fit the classifier

In [ ]:
species_model.fit(X_species_train, y_species_train)

### Step 5 — Predict species labels

In [ ]:
species_predictions = species_model.predict(X_species_test)
display(pd.DataFrame({
    "observed_species": y_species_test,
    "predicted_species": species_predictions,
}).head(10))

### Step 6 — Evaluate classification predictions
Accuracy is the proportion correctly classified. Macro F1 averages the class-specific F1 scores, giving each species equal weight. Precision asks how often a predicted species is correct; recall asks how many members of that species were identified. The baseline always predicts the most frequent training species.

In [ ]:
species_baseline = DummyClassifier(strategy="most_frequent")
species_baseline.fit(X_species_train, y_species_train)
baseline_species_predictions = species_baseline.predict(X_species_test)

def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

display(pd.DataFrame({
    "Most-frequent baseline": classification_metrics(y_species_test, baseline_species_predictions),
    "RidgeClassifier (alpha=1)": classification_metrics(y_species_test, species_predictions),
}).T.round(3))
print(classification_report(y_species_test, species_predictions, zero_division=0))

### Step 7 — View errors by species
Rows indicate observed species; columns indicate predicted species. Diagonal entries are correct predictions.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_species_test, species_predictions,
    labels=species_model.named_steps["ridge"].classes_,
    cmap="Blues", colorbar=False,
)
plt.title("RidgeClassifier: test confusion matrix")
plt.tight_layout()
plt.show()

### Step 8 — Inspect class scores
For each row, the classifier selects the species with the highest score. Scores may be negative and do not sum to one. `RidgeClassifier` does not offer `predict_proba()`.

In [ ]:
species_scores = pd.DataFrame(
    species_model.decision_function(X_species_test),
    columns=species_model.named_steps["ridge"].classes_,
    index=X_species_test.index,
)
display(species_scores.head().round(3))

## 4. Cross-validation for both prediction types
### What does five-fold cross-validation do?
Split the **training set** into five folds. Fit on four folds and validate on the fifth; repeat until every fold has served as validation. Summarize the five validation scores.

We first assess the fixed `alpha=1` models, then tune `alpha`. Passing the **whole pipeline** to cross-validation ensures imputation and scaling are learned separately within each training fold. These examples assume independent observations from a similar population; grouped or time-dependent data need an appropriate splitter.

### Regression: Step 1 — Define shuffled folds
Use `KFold` for a continuous target. Shuffling avoids dependence on the dataset's original row order.

In [ ]:
mass_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### Regression: Step 2 — Score the fixed model
`cross_validate` clones and refits the pipeline for every fold. Scikit-learn expresses losses as negative scores because its selection tools maximize scores. We reverse the sign for readable RMSE and MAE.

In [ ]:
mass_cv_results = cross_validate(
    clone(mass_model), X_mass_train, y_mass_train,
    cv=mass_cv,
    scoring={"RMSE": "neg_root_mean_squared_error",
             "MAE": "neg_mean_absolute_error", "R2": "r2"},
)
mass_fold_scores = pd.DataFrame({
    "RMSE_g": -mass_cv_results["test_RMSE"],
    "MAE_g": -mass_cv_results["test_MAE"],
    "R_squared": mass_cv_results["test_R2"],
}, index=pd.Index(range(1, 6), name="fold"))
display(mass_fold_scores.round(3))
display(mass_fold_scores.agg(["mean", "std"]).round(3))

Here `test_RMSE` means the **validation fold** score, not the held-out test set. The mean is the average of fold RMSEs; the standard deviation describes fold-to-fold variation and is not a confidence interval.

### Classification: Step 1 — Define stratified folds
`StratifiedKFold` approximately preserves the species proportions in each fold. Each class needs at least five training observations for this five-fold example.

In [ ]:
species_cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=RANDOM_STATE
)

### Classification: Step 2 — Score the fixed classifier

In [ ]:
species_cv_results = cross_validate(
    clone(species_model), X_species_train, y_species_train,
    cv=species_cv, scoring={"accuracy": "accuracy", "macro_F1": "f1_macro"},
)
species_fold_scores = pd.DataFrame({
    "accuracy": species_cv_results["test_accuracy"],
    "macro_F1": species_cv_results["test_macro_F1"],
}, index=pd.Index(range(1, 6), name="fold"))
display(species_fold_scores.round(3))
display(species_fold_scores.agg(["mean", "std"]).round(3))

## 5. Select the ridge penalty using cross-validation
### Step 1 — Specify candidate values
`np.logspace(-3, 3, 13)` creates 13 values from 0.001 to 1000 on a logarithmic scale. `ridge__alpha` means the `alpha` parameter of the pipeline step named `ridge`.

`GridSearchCV` evaluates every candidate on the same folds and, with `refit=True`, refits the best pipeline on all training rows. We wrap the entire pipeline in the search so preprocessing remains inside the folds.

In [ ]:
alpha_grid = np.logspace(-3, 3, 13)
parameter_grid = {"ridge__alpha": alpha_grid}
print(alpha_grid)

### Step 2 — Tune the regression model
Maximizing negative RMSE is equivalent to minimizing RMSE.

In [ ]:
mass_search = GridSearchCV(
    estimator=clone(mass_model), param_grid=parameter_grid,
    scoring="neg_root_mean_squared_error", cv=mass_cv, refit=True,
)
mass_search.fit(X_mass_train, y_mass_train)
print("Selected alpha:", mass_search.best_params_["ridge__alpha"])
print("Best mean CV RMSE (g):", round(-mass_search.best_score_, 2))

In [ ]:
mass_search_table = pd.DataFrame({
    "alpha": mass_search.cv_results_["param_ridge__alpha"].astype(float),
    "mean_CV_RMSE_g": -mass_search.cv_results_["mean_test_score"],
    "SD_CV_RMSE_g": mass_search.cv_results_["std_test_score"],
})
display(mass_search_table.round(3))

### Step 3 — Tune the classification model
Here we select the penalty by accuracy. Use `scoring="f1_macro"` instead if equal-weight performance across species is your primary goal; choose the metric before examining test results.

In [ ]:
species_search = GridSearchCV(
    estimator=clone(species_model), param_grid=parameter_grid,
    scoring="accuracy", cv=species_cv, refit=True,
)
species_search.fit(X_species_train, y_species_train)
print("Selected alpha:", species_search.best_params_["ridge__alpha"])
print("Best mean CV accuracy:", round(species_search.best_score_, 3))

In [ ]:
species_search_table = pd.DataFrame({
    "alpha": species_search.cv_results_["param_ridge__alpha"].astype(float),
    "mean_CV_accuracy": species_search.cv_results_["mean_test_score"],
    "SD_CV_accuracy": species_search.cv_results_["std_test_score"],
})
display(species_search_table.round(3))

### Step 4 — Visualize the tuning results
Look for a range of good penalties, not just a single winning value. If the best value falls at the edge of the grid, consider expanding the grid using training CV only.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogx(mass_search_table["alpha"], mass_search_table["mean_CV_RMSE_g"], marker="o")
axes[0].set(xlabel="Alpha", ylabel="Mean CV RMSE (g)", title="Body mass: lower is better")
axes[1].semilogx(species_search_table["alpha"], species_search_table["mean_CV_accuracy"], marker="o")
axes[1].set(xlabel="Alpha", ylabel="Mean CV accuracy", title="Species: higher is better")
plt.tight_layout()
plt.show()

### Step 5 — Evaluate the tuned models on held-out test data
The searches have already refitted their winning pipelines. Calling `.predict()` on the search uses that fitted winner. The test set never enters the search.

For teaching, we displayed the fixed-model test scores earlier. Do not use those results to change the grid, features, or scoring rule. In a formal analysis, finish model development before revealing any test scores. The winning CV score can be optimistic because it was used for selection; use the reserved test set for final evaluation, or use nested CV if you need a cross-validated estimate of the full tuning procedure.

In [ ]:
tuned_mass_predictions = mass_search.predict(X_mass_test)
tuned_species_predictions = species_search.predict(X_species_test)
display(pd.DataFrame({
    "Tuned ridge": regression_metrics(y_mass_test, tuned_mass_predictions)
}).T.round(3))
display(pd.DataFrame({
    "Tuned ridge classifier": classification_metrics(y_species_test, tuned_species_predictions)
}).T.round(3))